In [1]:
import os,re,json,gc,time,warnings,zipfile
from pathlib import Path
os.environ.update(HF_HUB_OFFLINE="1",TRANSFORMERS_OFFLINE="1",TOKENIZERS_PARALLELISM="true")
warnings.filterwarnings("ignore")

import numpy as np,pandas as pd,pyarrow.parquet as pq,torch
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer,AutoModelForSequenceClassification
from IPython.display import FileLink,display

D=Path("/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items")
M=Path("/kaggle/input/models/f1aver/ecup-product-matching-bge-v3/pytorch/default/1")
O=Path("/kaggle/working/egor_private_hypotheses");O.mkdir(parents=True,exist_ok=True)
L,I=D/"matches_llm.parquet",D/"items.parquet"
SEEDS=[13,29,47,71,97]; BS=128; MAXLEN=320; N_SIM=500; GRAY_REF=.5547
assert torch.cuda.device_count()==2 and L.exists() and I.exists() and (M/"model.safetensors").exists()
T=time.time();print("GPU:",[torch.cuda.get_device_name(i) for i in range(2)],"\nMODEL:",M,flush=True)

K=["бренд","артикул","партномер","oem","код","модель","размер","цвет","объем","обьем","вес","тип","материал","количество"]
C2L=str.maketrans("аеорсухАЕОРСУХКМТВНЗЅІі","aeopcyxAEOPCYXKMTBH3SIi")
L2C=str.maketrans("aeopcyxAEOPCYX","аеорсухАЕОРСУХ")
UR=re.compile(r"(\d+(?:[.,]\d+)?)\s*(мл|ml|л|l|мг|mg|г|g|гр|кг|kg|мм|mm|см|cm|м|мб|mb|гб|gb|тб|tb|вт|w|квт|kw|мач|mah)\b",re.I)
UM={"мл":("ml",1),"ml":("ml",1),"л":("ml",1000),"l":("ml",1000),"мг":("g",.001),"mg":("g",.001),"г":("g",1),"g":("g",1),"гр":("g",1),"кг":("g",1000),"kg":("g",1000),"мм":("mm",1),"mm":("mm",1),"см":("mm",10),"cm":("mm",10),"м":("mm",1000),"мб":("gb",.001),"mb":("gb",.001),"гб":("gb",1),"gb":("gb",1),"тб":("gb",1024),"tb":("gb",1024),"вт":("w",1),"w":("w",1),"квт":("w",1000),"kw":("w",1000),"мач":("mah",1),"mah":("mah",1)}
QR=[re.compile(r"(\d+)\s*шт"),re.compile(r"набор\w*\s+из\s+(\d+)"),re.compile(r"(\d+)\s*(?:набор|упаков|комплект)\w*\s+по\s+(\d+)"),re.compile(r"[xх*](\d+)\b")]

def text(n,a):
    p=[str(n) if n is not None else ""]
    try:d=json.loads(a) if isinstance(a,str) else {}
    except:d={}
    if isinstance(d,dict) and d:
        low={str(k).lower():str(v) for k,v in d.items() if v};pick=[];used=set()
        for w in K:
            for k,v in low.items():
                if w in k and k not in used:pick.append(f"{k}:{v}");used.add(k)
        p.append(" ; ".join(pick+[f"{k}:{v}" for k,v in low.items() if k not in used])[:520])
    s=" | ".join(p).replace("ё","е").replace("Ё","Е");z=[]
    for q in s.split():
        c=sum("\u0400"<=x<="\u04ff" for x in q);l=sum(x.isascii() and x.isalpha() for x in q)
        z.append(q.translate(C2L if l>=c else L2C) if c and l else q)
    s=re.sub(r"[×хХ](?=\d)","x"," ".join(z));u=set();e=[]
    for m in UR.finditer(s):
        un,k=UM[m.group(2).lower()];u.add(f"{float(m.group(1).replace(',','.'))*k:g}{un}")
    if u:e.append("ед: "+" ".join(sorted(u)[:12]))
    m=QR[2].search(s.lower())
    qty=int(m.group(1))*int(m.group(2)) if m else next((int(x.group(1)) for r in [QR[0],QR[1],QR[3]] if (x:=r.search(s.lower()))),None)
    if qty and 1<qty<=1000:e.append(f"кол-во: {qty}")
    return (s+(" | "+" | ".join(e) if e else ""))[:2000]

def components(a,b):
    parent={}
    def find(x):
        x=int(x);r=parent.setdefault(x,x)
        while r!=parent[r]:parent[r]=parent[parent[r]];r=parent[r]
        parent[x]=r;return r
    for x,y in zip(a,b):
        x,y=find(x),find(y)
        if x!=y:parent[y]=x
    return np.fromiter((find(x) for x in a),np.int64,len(a))

print("Читаем LLM и строим компоненты...",flush=True)
ll=pd.read_parquet(L,columns=["id1","id2","target"])
comp=components(ll.id1.values,ll.id2.values)
roots,inv=np.unique(comp,return_inverse=True)
gray=(ll.target.values>.2)&(ll.target.values<.8)
masks={};any_mask=np.zeros(len(ll),bool)

for seed in SEEDS:
    chosen=np.random.RandomState(seed).rand(len(roots))<.03
    masks[seed]=gray&chosen[inv];any_mask|=masks[seed]
    print("seed",seed,"gray",int(masks[seed].sum()),flush=True)

idx=np.flatnonzero(any_mask)
g=ll.iloc[idx].copy().reset_index(drop=True)
g["target_soft"]=g.target.astype(np.float32)
g["target"]=(g.target>=.5).astype(np.int8)
g["component"]=comp[idx]
for seed in SEEDS:g[f"seed_{seed}"]=masks[seed][idx]
del ll,comp,roots,inv,masks,gray,any_mask;gc.collect()
print("gray union:",len(g),flush=True)

need=set(g.id1)|set(g.id2);tx={};cats={}
for batch in pq.ParquetFile(I).iter_batches(columns=["id","name","attributes","category"],batch_size=400000):
    d=batch.to_pandas();d=d[d.id.isin(need)]
    for i,n,a,c in d.itertuples(index=False,name=None):
        i=int(i);tx[i]=text(n,a);cats[i]=str(c);need.discard(i)
    if not need:break
assert not need,f"Не найдено товаров: {len(need)}"
g["category"]=[cats[int(i)] for i in g.id1]
print("texts:",len(tx),flush=True)

tok=AutoTokenizer.from_pretrained(M,local_files_only=True)
raw=AutoModelForSequenceClassification.from_pretrained(M,local_files_only=True,dtype=torch.float16).cuda().eval()
model=torch.nn.DataParallel(raw,[0,1]).eval()

@torch.inference_mode()
def predict(df):
    cache=O/"gray_scores.npz"
    if cache.exists():
        z=np.load(cache);return z["p"],z["gap"]
    order=np.argsort([len(tx[int(a)])+len(tx[int(b)]) for a,b in zip(df.id1,df.id2)])
    p=np.empty(len(df),np.float32);gap=np.empty(len(df),np.float32);t=time.time()
    for s in range(0,len(order),BS):
        ix=order[s:s+BS];a=[tx[int(df.id1.iloc[i])] for i in ix];b=[tx[int(df.id2.iloc[i])] for i in ix]
        x=tok(a,b,padding=True,truncation=True,max_length=MAXLEN,pad_to_multiple_of=8,return_tensors="pt")
        y=tok(b,a,padding=True,truncation=True,max_length=MAXLEN,pad_to_multiple_of=8,return_tensors="pt")
        x={k:v.cuda() for k,v in x.items()};y={k:v.cuda() for k,v in y.items()}
        with torch.autocast("cuda",dtype=torch.float16):
            pf=torch.sigmoid(model(**x,return_dict=False)[0].squeeze(-1).float())
            pr=torch.sigmoid(model(**y,return_dict=False)[0].squeeze(-1).float())
        pf,pr=pf.cpu().numpy(),pr.cpu().numpy()
        p[ix]=(pf+pr)/2;gap[ix]=abs(pf-pr)
        if s==0 or s//BS%100==0:print(f"BGE {s+len(ix):,}/{len(df):,} {(s+len(ix))/max(1,time.time()-t):.0f}/s",flush=True)
    np.savez(cache,p=p,gap=gap);return p,gap

p,swap_gap=predict(g)
yv=g.target.values;cv=g.category.values

def metric(mask,details=False):
    scores={}
    for c in np.unique(cv[mask]):
        q=mask&(cv==c)
        if q.sum() and len(np.unique(yv[q]))==2:scores[c]=average_precision_score(yv[q],p[q])
    value=float(np.mean(list(scores.values()))) if len(scores)==20 else np.nan
    return (value,scores) if details else value

seed_rows=[];cat_rows=[]
for seed in SEEDS:
    m=g[f"seed_{seed}"].values
    score,detail=metric(m,True)
    seed_rows.append({"seed":seed,"pairs":int(m.sum()),"macro_pr_auc":score})
    for c,v in detail.items():
        q=m&(cv==c)
        cat_rows.append({"seed":seed,"category":c,"pairs":int(q.sum()),"positives":int(yv[q].sum()),"ap":v})

seed_df=pd.DataFrame(seed_rows)
raw_cat=pd.DataFrame(cat_rows)
cat_df=raw_cat.groupby("category").agg(
    pairs_mean=("pairs","mean"),positives_mean=("positives","mean"),
    ap_mean=("ap","mean"),ap_std=("ap","std"),ap_min=("ap","min"),ap_max=("ap","max")
).reset_index()
cat_df["ap_range"]=cat_df.ap_max-cat_df.ap_min
cat_df=cat_df.sort_values("ap_range",ascending=False)

# 500 псевдо-public/private разбиений на объединённой серой зоне.
uc,ci=np.unique(g.component.values,return_inverse=True)
blocks=[]
for c in sorted(np.unique(cv)):
    ix=np.flatnonzero(cv==c);ix=ix[np.argsort(-p[ix])]
    blocks.append((c,yv[ix],ci[ix]))

def ap_sorted(y,sel):
    z=y[sel];npos=int(z.sum())
    if not npos or npos==len(z):return np.nan
    pos=np.flatnonzero(z==1)
    return float(np.mean(np.cumsum(z)[pos]/(pos+1)))

full=metric(np.ones(len(g),bool))
rng=np.random.RandomState(20260827);sim=[]
while len(sim)<N_SIM:
    pub_comp=rng.rand(len(uc))<.30;pa=[];pr=[]
    for _,y,cix in blocks:
        pa.append(ap_sorted(y,pub_comp[cix]));pr.append(ap_sorted(y,~pub_comp[cix]))
    if np.isfinite(pa).all() and np.isfinite(pr).all():
        a,b=float(np.mean(pa)),float(np.mean(pr))
        sim.append({"public":a,"private":b,"delta":b-a,
                    "public_minus_full":a-full,"private_minus_full":b-full})
sim=pd.DataFrame(sim)

s13=float(seed_df.loc[seed_df.seed==13,"macro_pr_auc"].iloc[0])
public_abs=np.abs(sim.public_minus_full)
fragile=cat_df[(cat_df.ap_std>.02)|(cat_df.ap_range>.05)]

report={
 "model":"BGE V3, public LB 0.5535",
 "gray_union_pairs":len(g),
 "gray_union_macro":full,
 "gray_seed13":s13,
 "egor_gray_reference":GRAY_REF,
 "reference_error":s13-GRAY_REF,
 "reference_reproduced":bool(abs(s13-GRAY_REF)<=.003),
 "holdout_seed_mean":float(seed_df.macro_pr_auc.mean()),
 "holdout_seed_std":float(seed_df.macro_pr_auc.std()),
 "holdout_seed_range":float(seed_df.macro_pr_auc.max()-seed_df.macro_pr_auc.min()),
 "public_noise_abs_median":float(public_abs.median()),
 "public_noise_abs_p90":float(public_abs.quantile(.90)),
 "public_noise_abs_p95":float(public_abs.quantile(.95)),
 "p_private_below_public_005":float((sim.delta<-.005).mean()),
 "p_private_below_public_010":float((sim.delta<-.010).mean()),
 "private_delta_p05":float(sim.delta.quantile(.05)),
 "private_delta_median":float(sim.delta.median()),
 "swap_gap_mean":float(swap_gap.mean()),
 "swap_gap_p95":float(np.quantile(swap_gap,.95)),
 "fragile_categories":fragile.category.tolist(),
 "limitations":[
   "Настоящие private labels недоступны: проверяется sampling variance, не скрытый distribution shift.",
   "Multi-seed обучения нельзя проверить без 3-5 независимо обученных BGE-checkpoint.",
   "Корреляцию gray_full с LB между архитектурами нельзя проверить по одной BGE V3."
 ]
}

risk=0
risk+=int(report["holdout_seed_std"]>.003)
risk+=int(report["public_noise_abs_p95"]>.005)
risk+=int(report["p_private_below_public_010"]>.10)
risk+=int(len(fragile)>=5)
report["risk_points"]=risk
report["risk"]="LOW" if risk<=1 else "MEDIUM" if risk<=3 else "HIGH"

seed_df.to_csv(O/"gray_seed_metrics.csv",index=False)
raw_cat.to_csv(O/"gray_category_by_seed.csv",index=False)
cat_df.to_csv(O/"gray_category_stability.csv",index=False)
sim.to_csv(O/"pseudo_public_private.csv",index=False)
g.assign(predict=p,swap_gap=swap_gap).to_parquet(O/"gray_predictions.parquet",index=False)
(O/"report.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))

print("\nSEED STABILITY\n",seed_df.to_string(index=False))
print("\nCATEGORY STABILITY\n",cat_df.to_string(index=False))
print("\nFINAL REPORT\n",json.dumps(report,ensure_ascii=False,indent=2))

Z=Path("/kaggle/working/egor_private_hypotheses.zip")
with zipfile.ZipFile(Z,"w",zipfile.ZIP_DEFLATED) as z:
    for f in O.glob("*"):
        if f.is_file():z.write(f,f.name)
print(f"\nSAVED {Z}; minutes={(time.time()-T)/60:.1f}")
display(FileLink(str(Z)))

GPU: ['Tesla T4', 'Tesla T4'] 
MODEL: /kaggle/input/models/f1aver/ecup-product-matching-bge-v3/pytorch/default/1
Читаем LLM и строим компоненты...
seed 13 gray 45831
seed 29 gray 43527
seed 47 gray 43058
seed 71 gray 92315
seed 97 gray 42576
gray union: 255367
texts: 376782


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

BGE 128/255,367 68/s
BGE 12,928/255,367 155/s
BGE 25,728/255,367 124/s
BGE 38,528/255,367 110/s
BGE 51,328/255,367 100/s
BGE 64,128/255,367 92/s
BGE 76,928/255,367 85/s
BGE 89,728/255,367 79/s
BGE 102,528/255,367 76/s
BGE 115,328/255,367 73/s
BGE 128,128/255,367 71/s
BGE 140,928/255,367 69/s
BGE 153,728/255,367 67/s
BGE 166,528/255,367 66/s
BGE 179,328/255,367 65/s
BGE 192,128/255,367 64/s
BGE 204,928/255,367 64/s
BGE 217,728/255,367 63/s
BGE 230,528/255,367 62/s
BGE 243,328/255,367 62/s

SEED STABILITY
  seed  pairs  macro_pr_auc
   13  45831      0.555224
   29  43527      0.582330
   47  43058      0.584705
   71  92315      0.594143
   97  42576      0.600360

CATEGORY STABILITY
                category  pairs_mean  positives_mean  ap_mean   ap_std   ap_min   ap_max  ap_range
                 Аптека      5338.0          1074.6 0.587336 0.083571 0.479246 0.672716  0.193470
      Ювелирные изделия       268.6            74.0 0.431653 0.063703 0.331446 0.494599  0.163153
             

/kaggle/working/egor_private_hypotheses.zip